<a href="https://colab.research.google.com/github/shefiramarizcha62-sudo/Dataset-Narkotika_027_199/blob/crawling/06_PREPROCESSING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Persiapan Google Colab dan clone GitHub

## 1.1 — Clone repository dari branch crawling

In [1]:
# Clone repository dari branch crawling
!git clone -b crawling --single-branch https://github.com/shefiramarizcha62-sudo/Dataset-Narkotika_027_199.git

# Masuk ke direktori repository
%cd Dataset-Narkotika_027_199

# Periksa isi folder
!ls
!ls data

Cloning into 'Dataset-Narkotika_027_199'...
remote: Enumerating objects: 258, done.
remote: Counting objects: 100% (258/258), done.
remote: Compressing objects: 100% (225/225), done.
remote: Total 258 (delta 114), reused 141 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (258/258), 10.00 MiB | 11.60 MiB/s, done.
Resolving deltas: 100% (114/114), done.
/content/Dataset-Narkotika_027_199
01_crawling_metadata.ipynb  04_ekstraksi_putusan.ipynb	PDF_Putusan
02_audit_pdf.ipynb	    05_validasi_dataset.ipynb	TXT_Putusan
03_download_pdf.ipynb	    data
dua_kandidat_pengganti.csv    hasil_uji_download.csv
hasil_audit_pdf.csv	      kandidat_50_putusan.csv
hasil_download_pdf.csv	      kandidat_pdf.csv
hasil_download_pengganti.csv  metadata_putusan.csv
hasil_ekstraksi.csv


## 1.2 — Import library

In [3]:
import os
import re
import glob
import joblib
import numpy as np
import pandas as pd

from collections import Counter
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 2. Load dan eksplorasi dataset

## 2.1 Cek STRUKTUR DATASET

In [5]:
txt_directory = "TXT_Putusan"

years = ["2023", "2024", "2025"]

for year in years:

    year_path = os.path.join(txt_directory, year)

    if not os.path.isdir(year_path):
        raise FileNotFoundError(
            f"Folder tahun {year} tidak ditemukan: {year_path}"
        )

    txt_files = glob.glob(
        os.path.join(year_path, "**", "*.txt"),
        recursive=True
    )

    print(f"Tahun {year}: {len(txt_files)} file TXT")

print("\nStruktur dataset:")
for root, dirs, files in os.walk(txt_directory):
    level = root.replace(txt_directory, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        if file.lower().endswith(".txt"):
            print(f"{indent}    {file}")

Tahun 2023: 17 file TXT
Tahun 2024: 19 file TXT
Tahun 2025: 14 file TXT

Struktur dataset:
TXT_Putusan/
    2024/
        putusan_1741_Pid_Sus_2024_PN_Sby.txt
        putusan_2071_Pid_Sus_2024_PN_Sby.txt
        putusan_898_Pid_Sus_2024_PN_Sby.txt
        putusan_1068_Pid_Sus_2024_PN_Sby.txt
        putusan_1629_Pid_Sus_2024_PN_Sby.txt
        putusan_1904_Pid_Sus_2024_PN_Sby.txt
        putusan_2127_Pid_Sus_2024_PN_Sby.txt
        putusan_2302_Pid_Sus_2024_PN_Sby.txt
        putusan_2050_Pid_Sus_2024_PN_Sby.txt
        putusan_2010_Pid_Sus_2024_PN_Sby.txt
        putusan_2017_Pid_Sus_2024_PN_Sby.txt
        putusan_1751_Pid_Sus_2024_PN_Sby.txt
        putusan_1968_Pid_Sus_2024_PN_Sby.txt
        putusan_891_Pid_Sus_2024_PN_Sby.txt
        putusan_2054_Pid_Sus_2024_PN_Sby.txt
        putusan_1901_Pid_Sus_2024_PN_Sby.txt
        putusan_2454_Pid_Sus_2024_PN_Sby.txt
        putusan_1846_Pid_Sus_2024_PN_Sby.txt
        putusan_1276_Pid_Sus_2024_PN_Sby.txt
    2025/
        putusan_1773_Pi

## 2.2 Extracting seluruh dokumen TXT

In [7]:
def read_txt_file(file_path):

    encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin-1"
    ]

    for encoding in encodings:

        try:
            with open(
                file_path,
                "r",
                encoding=encoding
            ) as file:

                return file.read()

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"Gagal membaca file: {file_path}"
    )


documents = []

for year in years:

    year_path = os.path.join(txt_directory, year)

    txt_files = sorted(
        glob.glob(
            os.path.join(year_path, "**", "*.txt"),
            recursive=True
        )
    )

    for file_path in txt_files:

        raw_text = read_txt_file(file_path)

        document = {
            "document_id": os.path.splitext(
                os.path.basename(file_path)
            )[0],
            "year": year,
            "file_name": os.path.basename(file_path),
            "file_path": file_path,
            "raw_text": raw_text
        }

        documents.append(document)


df = pd.DataFrame(documents)

print("Extracting selesai.")
print("Jumlah dokumen:", len(df))
print("Jumlah kolom:", len(df.columns))

display(
    df[[
        "document_id",
        "year",
        "file_name"
    ]].head()
)

Extracting selesai.
Jumlah dokumen: 50
Jumlah kolom: 5


,document_id,year,file_name
0,putusan_1024_Pid_Sus_2023_PN_Sby,2023,putusan_1024_Pid_Sus_2023_PN_Sby.txt
1,putusan_140_Pid_Sus_2023_PN_Sby,2023,putusan_140_Pid_Sus_2023_PN_Sby.txt
2,putusan_1512_Pid_Sus_2023_PN_Sby,2023,putusan_1512_Pid_Sus_2023_PN_Sby.txt
3,putusan_1880_Pid_Sus_2023_PN_Sby,2023,putusan_1880_Pid_Sus_2023_PN_Sby.txt
4,putusan_1982_Pid_Sus_2023_PN_Sby,2023,putusan_1982_Pid_Sus_2023_PN_Sby.txt


## 2.3 Pemeriksaan dataset

In [8]:
df["raw_text"] = df["raw_text"].fillna("").astype(str)

df["character_count"] = df["raw_text"].str.len()

df["word_count"] = df["raw_text"].apply(
    lambda text: len(text.split())
)

print("Jumlah seluruh dokumen:", len(df))

print(
    "\nJumlah dokumen kosong:",
    (df["raw_text"].str.strip() == "").sum()
)

print("\nDistribusi dokumen berdasarkan tahun:")
display(
    df.groupby("year").size().reset_index(
        name="jumlah_dokumen"
    )
)

print("\nStatistik panjang dokumen:")
display(
    df[["character_count", "word_count"]].describe()
)

print("\nContoh isi dokumen pertama:")
print(df.iloc[0]["raw_text"][:3000])

Jumlah seluruh dokumen: 50

Jumlah dokumen kosong: 0

Distribusi dokumen berdasarkan tahun:


,year,jumlah_dokumen
0,2023,17
1,2024,19
2,2025,14



Statistik panjang dokumen:


,character_count,word_count
count,50.000000,50.000000
mean,77247.680000,10736.820000
std,33792.843691,4762.487931
min,44283.000000,6017.000000
25%,56041.500000,7798.250000
50%,67146.000000,9336.000000
75%,82092.500000,11549.250000
max,199841.000000,28188.000000



Contoh isi dokumen pertama:
hkama
ahkamah Agung Repub
ahkamah Agung Republik Indonesia
mah Agung Republik Indonesia
blik Indonesi
Direktori Putusan Mahkamah Agung Republik Indonesia
putusan.mahkamahagung.go.id
Halaman 1 dari 60 Putusan Nomor 1024/Pid.Sus/2023/PN Sby 
 
P U T U S A N 
Nomor 1024/Pid.Sus/2023/PN Sby 
 
DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA 
 
Pengadilan Negeri Surabaya yang mengadili perkara pidana dengan 
acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai 
berikut dalam perkara Para Terdakwa : 
Terdakwa 1 
1.  Nama lengkap  
: Surya Aji Pratama Bin Darmawan 
2.  Tempat lahir  
: Mojokerto 
3.  Umur/Tanggal lahir  
: 31 Tahun / 24 Mei 1992 
4.  Jenis kelamin  
: Laki-laki 
5.  Kebangsaan  
: Indonesia 
6.  Tempat tinggal  
: Dsn. Kalijaring, RT 004 / RW 007, Ds. Mlirip, Kec. 
Jetis, 
Kab. 
Mojokerto 
(sesuai 
NIK 
3516162405920006) atau bertempat tinggal di Jl. 
Kebraon Indah Permai, Kel. Kebraon, Kec. Karang 
Pilang, Kota Surabaya 
7.

# 3. Preprocessing

## 3.1 Case folding

In [10]:
def case_folding(text):

    return text.lower()


df["casefold_text"] = df["raw_text"].apply(case_folding)

display(
    df[[
        "file_name",
        "raw_text",
        "casefold_text"
    ]].head()
)

,file_name,raw_text,casefold_text
0,putusan_1024_Pid_Sus_2023_PN_Sby.txt,hkama\nahkamah Agung Repub\nahkamah Agung Repu...,hkama\nahkamah agung repub\nahkamah agung repu...
1,putusan_140_Pid_Sus_2023_PN_Sby.txt,hkama\nahkamah Agung Repub\nahkamah Agung Repu...,hkama\nahkamah agung repub\nahkamah agung repu...
2,putusan_1512_Pid_Sus_2023_PN_Sby.txt,hkama\nahkamah Agung Repub\nahkamah Agung Repu...,hkama\nahkamah agung repub\nahkamah agung repu...
3,putusan_1880_Pid_Sus_2023_PN_Sby.txt,hkama\nahkamah Agung Repub\nahkamah Agung Repu...,hkama\nahkamah agung repub\nahkamah agung repu...
4,putusan_1982_Pid_Sus_2023_PN_Sby.txt,hkama\nahkamah Agung Repub\nahkamah Agung Repu...,hkama\nahkamah agung repub\nahkamah agung repu...


## 3.2 Cleaning

In [16]:
def cleaning_text(text):

    if not isinstance(text, str):
        return ""

    lines = text.splitlines()

    cleaned_lines = []
    inside_disclaimer = False

    for line in lines:

        line = line.strip()

        if not line:
            continue

        if re.search(
            r"\bdisclaimer\b",
            line,
            flags=re.IGNORECASE
        ):
            inside_disclaimer = True
            continue

        if inside_disclaimer:

            if re.search(
                r"^halaman\s+\d+",
                line,
                flags=re.IGNORECASE
            ):
                inside_disclaimer = False

            continue

        if re.search(
            r"^halaman\s+\d+(?:\s+dari\s+\d+)?",
            line,
            flags=re.IGNORECASE
        ):
            continue

        if re.search(
            r"^putusan nomor\s+\d+",
            line,
            flags=re.IGNORECASE
        ):
            continue

        noise_patterns = [
            r"direktori putusan mahkamah agung",
            r"kepaniteraan mahkamah agung",
            r"putusan\.mahkamahagung\.go\.id",
            r"mahkamah agung republik indonesia",
            r"^hkama$",
            r"^ahkamah(?:\s|$)",
            r"^repub$",
            r"^mah agung(?:\s|$)",
            r"^blik indonesi"
        ]

        if any(
            re.search(
                pattern,
                line,
                flags=re.IGNORECASE
            )
            for pattern in noise_patterns
        ):
            continue

        line = re.sub(
            r"\bM\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\b",
            "MENGADILI",
            line,
            flags=re.IGNORECASE
        )

        line = re.sub(
            r"[•▪●]",
            " ",
            line
        )

        line = re.sub(
            r"^\s*[-–—]+\s*$",
            " ",
            line
        )

        cleaned_lines.append(line)

    cleaned_text = " ".join(cleaned_lines)

    cleaned_text = re.sub(
        r"\s+",
        " ",
        cleaned_text
    ).strip()

    return cleaned_text


df["cleaned_text"] = df["casefold_text"].apply(
    cleaning_text
)

display(
    df[[
        "file_name",
        "casefold_text",
        "cleaned_text"
    ]].head()
)

,file_name,casefold_text,cleaned_text
0,putusan_1024_Pid_Sus_2023_PN_Sby.txt,hkama\nahkamah agung repub\nahkamah agung repu...,p u t u s a n nomor 1024/pid.sus/2023/pn sby d...
1,putusan_140_Pid_Sus_2023_PN_Sby.txt,hkama\nahkamah agung repub\nahkamah agung repu...,p u t u s a n nomor 140/pid.sus/2023/pn sby de...
2,putusan_1512_Pid_Sus_2023_PN_Sby.txt,hkama\nahkamah agung repub\nahkamah agung repu...,p u t u s a n nomor 1512/pid.sus/2023/pn sby d...
3,putusan_1880_Pid_Sus_2023_PN_Sby.txt,hkama\nahkamah agung repub\nahkamah agung repu...,p u t u s a n nomor 1880/pid.sus/2023/pn sby d...
4,putusan_1982_Pid_Sus_2023_PN_Sby.txt,hkama\nahkamah agung repub\nahkamah agung repu...,p u t u s a n nomor 1982/pid.sus/2023/pn sby d...


## 3.3 Normalization

In [15]:
number_words = (
    r"satu|dua|tiga|empat|lima|enam|tujuh|delapan|sembilan|"
    r"sepuluh|sebelas|belas|puluh|ratus|ribu|juta|milyar|"
    r"miliar|triliun|koma|nol|rupiah"
)


def normalization_text(text):

    if not isinstance(text, str):
        return ""

    pattern = (
        r"(?<=\d)\s*\("
        r"(?:" + number_words + r")"
        r"(?:\s+(?:" + number_words + r"))*"
        r"\)"
    )

    text = re.sub(
        pattern,
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


df["normalized_text"] = df["cleaned_text"].apply(
    normalization_text
)

display(
    df[[
        "file_name",
        "cleaned_text",
        "normalized_text"
    ]].head()
)

,file_name,cleaned_text,normalized_text
0,putusan_1024_Pid_Sus_2023_PN_Sby.txt,p u t u s a n nomor 1024/pid.sus/2023/pn sby d...,p u t u s a n nomor 1024/pid.sus/2023/pn sby d...
1,putusan_140_Pid_Sus_2023_PN_Sby.txt,p u t u s a n nomor 140/pid.sus/2023/pn sby de...,p u t u s a n nomor 140/pid.sus/2023/pn sby de...
2,putusan_1512_Pid_Sus_2023_PN_Sby.txt,p u t u s a n nomor 1512/pid.sus/2023/pn sby d...,p u t u s a n nomor 1512/pid.sus/2023/pn sby d...
3,putusan_1880_Pid_Sus_2023_PN_Sby.txt,p u t u s a n nomor 1880/pid.sus/2023/pn sby d...,p u t u s a n nomor 1880/pid.sus/2023/pn sby d...
4,putusan_1982_Pid_Sus_2023_PN_Sby.txt,p u t u s a n nomor 1982/pid.sus/2023/pn sby d...,p u t u s a n nomor 1982/pid.sus/2023/pn sby d...


## 3.4 Tokenization

In [17]:
def tokenize_text(text):

    if not isinstance(text, str):
        return []

    tokens = re.findall(
        r"\d+(?:[.,/-]\d+)*|[^\W\d_]+",
        text.lower(),
        flags=re.UNICODE
    )

    return tokens


df["tokens"] = df["normalized_text"].apply(tokenize_text)

# Perbandingan Before dan After
df["tokenized_text"] = df["tokens"].apply(
    lambda tokens: " ".join(tokens)
)

display(
    df[[
        "file_name",
        "normalized_text",
        "tokenized_text"
    ]].head()
)

,file_name,normalized_text,tokenized_text
0,putusan_1024_Pid_Sus_2023_PN_Sby.txt,p u t u s a n nomor 1024/pid.sus/2023/pn sby d...,p u t u s a n nomor 1024 pid sus 2023 pn sby d...
1,putusan_140_Pid_Sus_2023_PN_Sby.txt,p u t u s a n nomor 140/pid.sus/2023/pn sby de...,p u t u s a n nomor 140 pid sus 2023 pn sby de...
2,putusan_1512_Pid_Sus_2023_PN_Sby.txt,p u t u s a n nomor 1512/pid.sus/2023/pn sby d...,p u t u s a n nomor 1512 pid sus 2023 pn sby d...
3,putusan_1880_Pid_Sus_2023_PN_Sby.txt,p u t u s a n nomor 1880/pid.sus/2023/pn sby d...,p u t u s a n nomor 1880 pid sus 2023 pn sby d...
4,putusan_1982_Pid_Sus_2023_PN_Sby.txt,p u t u s a n nomor 1982/pid.sus/2023/pn sby d...,p u t u s a n nomor 1982 pid sus 2023 pn sby d...


## 3.5 Pembentukan teks hasil preprocessing

In [18]:
df["processed_text"] = df["tokens"].apply(
    lambda tokens: " ".join(tokens)
)

df["word_count_before"] = df["raw_text"].apply(
    lambda text: len(text.split())
)

df["word_count_after"] = df["processed_text"].apply(
    lambda text: len(text.split())
)

df["character_count_before"] = df["raw_text"].str.len()
df["character_count_after"] = df["processed_text"].str.len()

display(
    df[[
        "file_name",
        "word_count_before",
        "word_count_after",
        "character_count_before",
        "character_count_after",
        "processed_text"
    ]].head()
)

,file_name,word_count_before,word_count_after,character_count_before,character_count_after,processed_text
0,putusan_1024_Pid_Sus_2023_PN_Sby.txt,26721,18666,190739,123641,p u t u s a n nomor 1024 pid sus 2023 pn sby d...
1,putusan_140_Pid_Sus_2023_PN_Sby.txt,8673,5832,63029,39502,p u t u s a n nomor 140 pid sus 2023 pn sby de...
2,putusan_1512_Pid_Sus_2023_PN_Sby.txt,8413,5928,59918,38644,p u t u s a n nomor 1512 pid sus 2023 pn sby d...
3,putusan_1880_Pid_Sus_2023_PN_Sby.txt,28188,20280,199841,134292,p u t u s a n nomor 1880 pid sus 2023 pn sby d...
4,putusan_1982_Pid_Sus_2023_PN_Sby.txt,6310,4308,45597,29023,p u t u s a n nomor 1982 pid sus 2023 pn sby d...


## 3.8 Pemeriksaan hasil preprocessing

In [19]:
preprocessing_summary = pd.DataFrame({
    "Keterangan": [
        "Jumlah dokumen",
        "Dokumen kosong sebelum preprocessing",
        "Dokumen kosong setelah preprocessing",
        "Rata-rata kata sebelum preprocessing",
        "Rata-rata kata setelah preprocessing",
        "Rata-rata karakter sebelum preprocessing",
        "Rata-rata karakter setelah preprocessing"
    ],
    "Nilai": [
        len(df),
        (df["raw_text"].str.strip() == "").sum(),
        (df["processed_text"].str.strip() == "").sum(),
        df["word_count_before"].mean(),
        df["word_count_after"].mean(),
        df["character_count_before"].mean(),
        df["character_count_after"].mean()
    ]
})

display(preprocessing_summary)

,Keterangan,Nilai
0,Jumlah dokumen,50.00
1,Dokumen kosong sebelum preprocessing,0.00
2,Dokumen kosong setelah preprocessing,0.00
3,Rata-rata kata sebelum preprocessing,10736.82
4,Rata-rata kata setelah preprocessing,7435.86
5,Rata-rata karakter sebelum preprocessing,77247.68
6,Rata-rata karakter setelah preprocessing,49665.22


# 4. INDEXING

## 4.1 TF-IDF Indexing

In [20]:
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(
    df["processed_text"]
)

print("TF-IDF indexing berhasil.")

print("Jumlah dokumen:", tfidf_matrix.shape[0])
print("Jumlah fitur:", tfidf_matrix.shape[1])
print("Dimensi matriks:", tfidf_matrix.shape)

TF-IDF indexing berhasil.
Jumlah dokumen: 50
Jumlah fitur: 42069
Dimensi matriks: (50, 42069)


## 4.2 Pemeriksaan hasil pembobotan TF-IDF

In [21]:
feature_names = vectorizer.get_feature_names_out()

document_index = 0

tfidf_values = tfidf_matrix[
    document_index
].toarray().flatten()

top_indices = np.argsort(tfidf_values)[::-1][:20]

tfidf_result = pd.DataFrame({
    "Term": feature_names[top_indices],
    "TF-IDF Weight": tfidf_values[top_indices]
})

display(tfidf_result)

,Term,TF-IDF Weight
0,aji,0.053798
1,aji pratama,0.053748
2,surya aji,0.053748
3,kebraon,0.052308
4,wahyuning,0.051391
5,wahyuning gusti,0.051391
6,satriyo wahyuning,0.049996
7,satriyo,0.049996
8,bin darmawan,0.048514
9,terdakwa surya,0.047667


## 4.3 fungsi pencarian dokumen

In [22]:
def search_documents(query, top_n=5):

    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query tidak boleh kosong.")

    query = query.lower()
    query = cleaning_text(query)
    query = normalization_text(query)

    query_tokens = tokenize_text(query)
    query_processed = " ".join(query_tokens)

    query_vector = vectorizer.transform([query_processed])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    ranked_indices = np.argsort(scores)[::-1]

    results = []

    for rank, idx in enumerate(
        ranked_indices[:top_n],
        start=1
    ):

        results.append({
            "Rank": rank,
            "Document ID": df.iloc[idx]["document_id"],
            "Year": df.iloc[idx]["year"],
            "File Name": df.iloc[idx]["file_name"],
            "Similarity Score": scores[idx]
        })

    return pd.DataFrame(results)

## 4.4 Pengujian pencarian dokumen

In [23]:
query = "narkotika jenis sabu dengan berat lebih dari 5 gram"

search_results = search_documents(
    query,
    top_n=5
)

display(search_results)

,Rank,Document ID,Year,File Name,Similarity Score
0,1,putusan_1789_Pid_Sus_2025_PN_Sby,2025,putusan_1789_Pid_Sus_2025_PN_Sby.txt,0.061078
1,2,putusan_2071_Pid_Sus_2024_PN_Sby,2024,putusan_2071_Pid_Sus_2024_PN_Sby.txt,0.060514
2,3,putusan_1668_Pid_Sus_2025_PN_Sby,2025,putusan_1668_Pid_Sus_2025_PN_Sby.txt,0.054790
3,4,putusan_2166_Pid_Sus_2023_PN_Sby,2023,putusan_2166_Pid_Sus_2023_PN_Sby.txt,0.054368
4,5,putusan_1982_Pid_Sus_2023_PN_Sby,2023,putusan_1982_Pid_Sus_2023_PN_Sby.txt,0.045335


In [24]:
queries = [
    "narkotika jenis sabu",
    "narkotika jenis ganja",
    "pemufakatan jahat narkotika",
    "tanpa hak atau melawan hukum",
    "pidana penjara dan denda",
    "barang bukti dirampas untuk dimusnahkan"
]

all_search_results = []

for query in queries:

    results = search_documents(query, top_n=5)

    results.insert(0, "Query", query)

    all_search_results.append(results)

search_evaluation = pd.concat(
    all_search_results,
    ignore_index=True
)

display(search_evaluation)

,Query,Rank,Document ID,Year,File Name,Similarity Score
0,narkotika jenis sabu,1,putusan_1846_Pid_Sus_2024_PN_Sby,2024,putusan_1846_Pid_Sus_2024_PN_Sby.txt,0.052754
1,narkotika jenis sabu,2,putusan_1982_Pid_Sus_2023_PN_Sby,2023,putusan_1982_Pid_Sus_2023_PN_Sby.txt,0.052462
2,narkotika jenis sabu,3,putusan_1901_Pid_Sus_2024_PN_Sby,2024,putusan_1901_Pid_Sus_2024_PN_Sby.txt,0.048992
3,narkotika jenis sabu,4,putusan_2031_Pid_Sus_2023_PN_Sby,2023,putusan_2031_Pid_Sus_2023_PN_Sby.txt,0.048665
4,narkotika jenis sabu,5,putusan_1741_Pid_Sus_2024_PN_Sby,2024,putusan_1741_Pid_Sus_2024_PN_Sby.txt,0.047770
5,narkotika jenis ganja,1,putusan_2043_Pid_Sus_2023_PN_Sby,2023,putusan_2043_Pid_Sus_2023_PN_Sby.txt,0.082529
6,narkotika jenis ganja,2,putusan_579_Pid_Sus_2025_PN_Sby,2025,putusan_579_Pid_Sus_2025_PN_Sby.txt,0.077739
7,narkotika jenis ganja,3,putusan_2325_Pid_Sus_2023_PN_Sby,2023,putusan_2325_Pid_Sus_2023_PN_Sby.txt,0.072675
8,narkotika jenis ganja,4,putusan_1773_Pid_Sus_2025_PN_Sby,2025,putusan_1773_Pid_Sus_2025_PN_Sby.txt,0.065945
9,narkotika jenis ganja,5,putusan_2054_Pid_Sus_2024_PN_Sby,2024,putusan_2054_Pid_Sus_2024_PN_Sby.txt,0.047787


# 5. Menyimpan hasil preprocessing dan indexing

In [25]:
output_directory = "/content/output_subcpmk4"

os.makedirs(
    output_directory,
    exist_ok=True
)

df.to_csv(
    os.path.join(
        output_directory,
        "dataset_preprocessed.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

joblib.dump(
    tfidf_matrix,
    os.path.join(
        output_directory,
        "tfidf_matrix.pkl"
    )
)

joblib.dump(
    vectorizer,
    os.path.join(
        output_directory,
        "tfidf_vectorizer.pkl"
    )
)

pd.DataFrame({
    "term": vectorizer.get_feature_names_out()
}).to_csv(
    os.path.join(
        output_directory,
        "tfidf_terms.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

search_evaluation.to_csv(
    os.path.join(
        output_directory,
        "search_results.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print("Seluruh hasil berhasil disimpan.")

display(
    pd.DataFrame({
        "Nama File": os.listdir(output_directory)
    })
)

Seluruh hasil berhasil disimpan.


,Nama File
0,dataset_preprocessed.csv
1,search_results.csv
2,tfidf_vectorizer.pkl
3,tfidf_terms.csv
4,tfidf_matrix.pkl


In [28]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/output_subcpmk4",
    "zip",
    output_directory
)

display(
    pd.DataFrame({
        "Nama File": ["output_subcpmk4.zip"],
        "Lokasi": [zip_path],
        "Ukuran (MB)": [
            round(os.path.getsize(zip_path) / (1024 * 1024), 2)
        ]
    })
)

files.download(zip_path)

,Nama File,Lokasi,Ukuran (MB)
0,output_subcpmk4.zip,/content/output_subcpmk4.zip,3.54


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 6. Ringkasan hasil pengerjaan

In [27]:
summary = pd.DataFrame({
    "Komponen": [
        "Jumlah dokumen",
        "Dokumen tahun 2023",
        "Dokumen tahun 2024",
        "Dokumen tahun 2025",
        "Jumlah dokumen kosong setelah preprocessing",
        "Metode preprocessing",
        "Metode pembobotan",
        "N-gram",
        "Jumlah fitur TF-IDF",
        "Dimensi matriks TF-IDF"
    ],
    "Hasil": [
        len(df),
        (df["year"] == "2023").sum(),
        (df["year"] == "2024").sum(),
        (df["year"] == "2025").sum(),
        (df["processed_text"].str.strip() == "").sum(),
        "Case Folding, Cleaning, Normalization, Tokenization",
        "TF-IDF dengan Sublinear TF",
        "Unigram dan Bigram",
        tfidf_matrix.shape[1],
        str(tfidf_matrix.shape)
    ]
})

display(summary)

,Komponen,Hasil
0,Jumlah dokumen,50
1,Dokumen tahun 2023,17
2,Dokumen tahun 2024,19
3,Dokumen tahun 2025,14
4,Jumlah dokumen kosong setelah preprocessing,0
5,Metode preprocessing,"Case Folding, Cleaning, Normalization, Tokeniz..."
6,Metode pembobotan,TF-IDF dengan Sublinear TF
7,N-gram,Unigram dan Bigram
8,Jumlah fitur TF-IDF,42069
9,Dimensi matriks TF-IDF,"(50, 42069)"
